In [ ]:
# ============================================
# Volleyball Nations League Analytics Platform
# Random Forest Classifier
# ============================================

# =====================================================
# Import Libraries
# =====================================================

import os
import joblib
import pandas as pd
import psycopg2

from dotenv import load_dotenv

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# =====================================================
# Connect to PostgreSQL
# =====================================================

load_dotenv()

connection = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    sslmode="require"
)

# =====================================================
# Load Training Data
# =====================================================

query = """
SELECT *
FROM ml_training_data
ORDER BY match_date, match_id;
"""

df = pd.read_sql_query(query, connection)

connection.close()

df["match_date"] = pd.to_datetime(df["match_date"])

print("=" * 60)
print("Dataset Loaded Successfully")
print("=" * 60)
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

display(df.head())

# =====================================================
# Feature Selection
# =====================================================

features = [
    "win_rate_diff",
    "attack_efficiency_diff",
    "attack_kills_diff",
    "serve_aces_diff",
    "serve_errors_diff",
    "serve_efficiency_diff",
    "reception_positive_diff",
    "reception_perfect_diff",
    "block_points_diff",
    "block_touches_diff",
    "digs_diff",
    "assists_diff",
    "points_diff",
    "break_points_diff"
]

X = df[features]
y = df["target"]

# =====================================================
# Train-Test Split
# Week 1 -> Training
# Week 2 -> Testing
# =====================================================

train = df[df["week"] == 1]
test = df[df["week"] == 2]

X_train = train[features]
y_train = train["target"]

X_test = test[features]
y_test = test["target"]

print("\nTraining Samples :", len(train))
print("Testing Samples  :", len(test))

# =====================================================
# Train Random Forest Model
# =====================================================

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

print("\nModel trained successfully!")

# =====================================================
# Save Model
# =====================================================

os.makedirs("../models", exist_ok=True)

joblib.dump(
    model,
    "../models/random_forest.pkl"
)

print("Model saved to ../models/random_forest.pkl")

# =====================================================
# Generate Predictions
# =====================================================

predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

# =====================================================
# Model Evaluation
# =====================================================

accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)
f1 = f1_score(y_test, predictions)
roc_auc = roc_auc_score(y_test, probabilities)

print("\n" + "=" * 60)
print("MODEL PERFORMANCE")
print("=" * 60)

print(f"Accuracy : {accuracy:.2%}")
print(f"Precision: {precision:.2%}")
print(f"Recall   : {recall:.2%}")
print(f"F1 Score : {f1:.2%}")
print(f"ROC-AUC  : {roc_auc:.2%}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test, predictions))

print("\nClassification Report")
print(classification_report(y_test, predictions))

# =====================================================
# Prediction Results
# =====================================================

prediction_results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": predictions,
    "Probability_Team_A_Wins": probabilities
})

display(prediction_results)

# =====================================================
# Feature Importance
# =====================================================

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print("\nFeature Importance (Random Forest)")
display(importance_df)

# =====================================================
# Save Outputs
# =====================================================

os.makedirs("../outputs", exist_ok=True)

prediction_results.to_csv(
    "../outputs/random_forest_predictions.csv",
    index=False
)

importance_df.to_csv(
    "../outputs/random_forest_feature_importance.csv",
    index=False
)

metrics = pd.DataFrame({
    "Model": ["Random Forest"],
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1 Score": [f1],
    "ROC-AUC": [roc_auc]
})

metrics.to_csv(
    "../outputs/random_forest_metrics.csv",
    index=False
)

print("\nOutputs saved successfully.")